# CPM nested LOOCV permutation test

本脚本只做 permutation test 和 p 值计算，不重新计算 observed nested CV。

Observed performance 来自 `CPM_nestedCV-LOOCV.ipynb` 已保存的：

- `CPM_nestedCV_highIAT_performance.pkl`
- `CPM_nestedCV_lowIAT_performance.pkl`

Permutation 使用与 observed nested LOOCV 完全一致的数据筛选、协变量、选边、inner CV 和 Ridge alpha 网格。

In [1]:
# ============================================================
# 0. 线程控制：需要在 numpy/scipy/sklearn 大量调用前设置
# ============================================================
import os

MAX_BLAS_THREADS = 1

os.environ["OMP_NUM_THREADS"] = str(MAX_BLAS_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(MAX_BLAS_THREADS)
os.environ["MKL_NUM_THREADS"] = str(MAX_BLAS_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(MAX_BLAS_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(MAX_BLAS_THREADS)

print("MAX_BLAS_THREADS=", MAX_BLAS_THREADS)

MAX_BLAS_THREADS= 1


In [2]:
# ============================================================
# 1. Imports：优先调用 DATA_ROOT/biox 中已经封装好的 permutation 工具
# ============================================================
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 数据和 biox package 的共同根目录。
DATA_ROOT = Path("/Users/fmri24/Desktop/YiZhang")

# biox package 位于 DATA_ROOT/biox；导入 package 时需要把父目录 DATA_ROOT 加入 sys.path。
if str(DATA_ROOT) not in sys.path:
    sys.path.insert(0, str(DATA_ROOT))

from biox.CPM import cpm_permutation_test_parallel, calculate_permutation_p_value
from biox.fcmat import edgeList
from biox.prep import fcMatrixGp

print("DATA_ROOT:", DATA_ROOT)
print("Imported biox.CPM.cpm_permutation_test_parallel and calculate_permutation_p_value")

DATA_ROOT: /Users/fmri24/Desktop/YiZhang
Imported biox.CPM.cpm_permutation_test_parallel and calculate_permutation_p_value


## 2. 路径与参数

`OBSERVED_RESULT_DIR` 用于读取 `CPM_nestedCV-LOOCV.ipynb` 已保存的 observed performance；`RESULT_DIR` 用于保存 permutation 输出。

In [3]:
# ============================================================
# 2. 路径与全局参数
# ============================================================
SCALE_CSV = DATA_ROOT / "量表数据汇总_with_MeanEnorm.csv"

SCHAEFER_DIR = DATA_ROOT / "HBN_func_Schaefer200"
HO_DIR = DATA_ROOT / "HBN_func_HarvardOxfordSubcortical"
HO_RAW_INDEX = np.array([4, 5, 6, 7, 9, 10, 11, 15, 16, 17, 18, 19, 20, 21]) - 1

IAT_LABEL = "IAT_PCIAT_Total"
SITE_COL = "ScannerSite"
CBCL_TOTAL_COL = "CBCL,CBCL_Total_T"
MOTION_COL = "mean_enorm"

COVARIATE_COLS = [
    "Basic_Demos,Age",
    "Basic_Demos,Sex",
    "Barratt,Barratt_Total_Edu",
    "Barratt,Barratt_Total_Occ",
    "CBCL,CBCL_Int_T",
    "CBCL,CBCL_Ext_T",
    MOTION_COL,
]

EXPECTED_N_TIMEPOINTS = 740
IAT_CUTOFF = 30
CBCL_CUTOFF = 70
ENORM_CUTOFF = 0.3
SUBGROUPS = ["highIAT", "lowIAT"]

# observed performance 来自 nested LOOCV 主分析；permutation 结果单独保存。
OBSERVED_RESULT_DIR = DATA_ROOT / "CPM_nestedCV_LOOCV"
RESULT_DIR = DATA_ROOT / "CPM_nestedCV_Permutation"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# CPM / nested CV 设置必须与 observed nested LOOCV 主分析一致。
P_THRESH = 0.01
INNER_N_SPLITS = 5
RANDOM_STATE = 2026
N_JOBS = 20
STANDARDIZE_FEATURES = True

ALPHA_GRID = np.logspace(-2, 2, 100)
ALPHA_GRID = np.sort(ALPHA_GRID)

# Permutation test 参数。
# nested LOOCV permutation 非常耗时；正式结果使用 5000 次。
N_PERMUTATIONS = 5000
PERMUTATION_METRIC = "pearson_r"
PERMUTATION_RANDOM_STATE = RANDOM_STATE + 1000

print("OBSERVED_RESULT_DIR:", OBSERVED_RESULT_DIR)
print("RESULT_DIR:", RESULT_DIR)
print("n_alpha:", len(ALPHA_GRID), "alpha range:", (ALPHA_GRID.min(), ALPHA_GRID.max()))
print("N_PERMUTATIONS:", N_PERMUTATIONS)

OBSERVED_RESULT_DIR: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_LOOCV
RESULT_DIR: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_Permutation
n_alpha: 100 alpha range: (np.float64(0.01), np.float64(100.0))
N_PERMUTATIONS: 5000


## 3. 准备 permutation 输入

Permutation 需要 `X/y/Covariates`，因此本节重新读取量表和功能时间序列，并按 observed nested LOOCV 脚本中的相同规则计算 FC。

本节不运行 observed CV，也不保存 observed 模型结果。

In [4]:
# ============================================================
# 3. FC 计算 / 数据读取
# ============================================================
# 这里与 CPM_nestedCV-LOOCV.ipynb 保持一致：
# 读取两个 run，拼接 Schaefer200 + HO14，计算 Fisher-z FC，再提取上三角 edge。

df_atlas = pd.read_csv(SCALE_CSV)
df_atlas["ID"] = df_atlas["ID"].astype(str)
df_atlas["func1"] = "sub-" + df_atlas["ID"] + "_func1.npy"
df_atlas["func2"] = "sub-" + df_atlas["ID"] + "_func2.npy"

files_present_in_both_atlases = set(os.listdir(SCHAEFER_DIR)) & set(os.listdir(HO_DIR))
df_atlas = df_atlas[
    df_atlas["func1"].isin(files_present_in_both_atlases)
    & df_atlas["func2"].isin(files_present_in_both_atlases)
].reset_index(drop=True)

fc_list = []
timepoint_ok_row_idx = []
for i in range(df_atlas.shape[0]):
    f1 = df_atlas.loc[i, "func1"]
    f2 = df_atlas.loc[i, "func2"]

    sch1 = np.load(SCHAEFER_DIR / f1)
    sch2 = np.load(SCHAEFER_DIR / f2)
    ho1 = np.load(HO_DIR / f1)
    ho2 = np.load(HO_DIR / f2)

    if (sch1.shape[1] + sch2.shape[1] != EXPECTED_N_TIMEPOINTS) or (ho1.shape[1] + ho2.shape[1] != EXPECTED_N_TIMEPOINTS):
        continue

    ts_sch = np.concatenate([sch1, sch2], axis=1)
    ts_ho = np.concatenate([ho1, ho2], axis=1)[HO_RAW_INDEX, :]
    ts = np.concatenate([ts_sch, ts_ho], axis=0)

    fc_z = fcMatrixGp(ts[np.newaxis, :, :], fisherZ=True)[0]
    fc_list.append(edgeList(fc_z, k=1))
    timepoint_ok_row_idx.append(i)

df_atlas = df_atlas.iloc[timepoint_ok_row_idx].reset_index(drop=True)
FC_all = np.asarray(fc_list, dtype=np.float32)
print("FC_all:", FC_all.shape)

/Users/fmri24/Desktop/YiZhang/biox/prep.py:68: RuntimeWarning: divide by zero encountered in arctanh
  fcMatrixGp = np.arctanh(fcMatrixGp)


FC_all: (1376, 22791)


## 4. 运行 permutation test 并计算 p 值

Observed score 从 `*_performance.pkl` 读取，不在本 notebook 重新估计。Permutation 每次都会重新运行完整 nested LOOCV，包括 inner KFold 选择 `ridge_alpha`。

In [5]:
# ============================================================
# 4. Permutation test：读取 observed pkl，计算 null distribution 和 p 值
# ============================================================
for subgroup in SUBGROUPS:
    df = df_atlas.copy()

    for c in COVARIATE_COLS + [IAT_LABEL, CBCL_TOTAL_COL]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    analysis_ready_mask = (
        df[COVARIATE_COLS + [IAT_LABEL, CBCL_TOTAL_COL, SITE_COL]].notna().all(axis=1).to_numpy()
        & np.isfinite(df[IAT_LABEL].to_numpy(dtype=float))
        & np.isfinite(df[MOTION_COL].to_numpy(dtype=float))
        & (df[MOTION_COL].to_numpy(dtype=float) < ENORM_CUTOFF)
        & (df[CBCL_TOTAL_COL].to_numpy(dtype=float) < CBCL_CUTOFF)
    )

    if subgroup == "highIAT":
        subgroup_mask = df[IAT_LABEL].to_numpy(dtype=float) > IAT_CUTOFF
    else:
        subgroup_mask = df[IAT_LABEL].to_numpy(dtype=float) <= IAT_CUTOFF

    subgroup_analysis_mask = analysis_ready_mask & subgroup_mask

    y_sub = df.loc[subgroup_analysis_mask, IAT_LABEL].to_numpy(dtype=float)
    X_sub = FC_all[subgroup_analysis_mask].astype(np.float32, copy=False)
    cov_basic = df.loc[subgroup_analysis_mask, COVARIATE_COLS].to_numpy(dtype=float)
    site_dummies = pd.get_dummies(
        df.loc[subgroup_analysis_mask, SITE_COL].astype(str),
        prefix=SITE_COL,
        drop_first=True,
    )
    cov_sub = np.column_stack([cov_basic, site_dummies.to_numpy(dtype=float)])

    finite_fc_mask = np.isfinite(X_sub).all(axis=1)
    X_sub = X_sub[finite_fc_mask]
    y_sub = y_sub[finite_fc_mask]
    cov_sub = cov_sub[finite_fc_mask]

    print(f"\n[{subgroup}] permutation N={len(y_sub)}, n_edges={X_sub.shape[1]}, covariates={cov_sub.shape[1]}")
    print(f"Running {N_PERMUTATIONS} permutations for {PERMUTATION_METRIC} with n_jobs={N_JOBS}")

    performance_path = OBSERVED_RESULT_DIR / f"CPM_nestedCV_{subgroup}_performance.pkl"
    if not performance_path.exists():
        raise FileNotFoundError(
            f"Observed performance pkl not found: {performance_path}. "
            "Please run CPM_nestedCV-LOOCV.ipynb first."
        )

    with open(performance_path, "rb") as f:
        observed_performance = pickle.load(f)

    observed_score = observed_performance["mean_prediction"][PERMUTATION_METRIC]

    permutation_result = cpm_permutation_test_parallel(
        X=X_sub,
        y=y_sub,
        Covariates=cov_sub,
        n_permutations=N_PERMUTATIONS,
        performance_metric=PERMUTATION_METRIC,
        use_mean_prediction=True,
        param_grid={"ridge_alpha": ALPHA_GRID},
        cv_type="loocv",
        n_repeats=1,
        inner_cv_type="kfold",
        inner_n_splits=INNER_N_SPLITS,
        scoring="pearson_r",
        selection_method="pvalue",
        p_thresh=P_THRESH,
        model_type="ridge",
        standardize_features=STANDARDIZE_FEATURES,
        random_state=PERMUTATION_RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=True,
    )

    p_value_result = calculate_permutation_p_value(
        observed_score=observed_score,
        null_scores=permutation_result["null_scores"],
        performance_metric=PERMUTATION_METRIC,
        alternative="auto",
    )

    permutation_path = RESULT_DIR / f"CPM_nestedCV_{subgroup}_permutation.pkl"
    p_value_path = RESULT_DIR / f"CPM_nestedCV_{subgroup}_permutation_p_value.pkl"
    null_csv_path = RESULT_DIR / f"CPM_nestedCV_{subgroup}_permutation_null_scores.csv"

    with open(permutation_path, "wb") as f:
        pickle.dump(permutation_result, f)
    with open(p_value_path, "wb") as f:
        pickle.dump(p_value_result, f)
    pd.DataFrame({PERMUTATION_METRIC: permutation_result["null_scores"]}).to_csv(null_csv_path, index=False)

    print("Observed score:", observed_score)
    print("Permutation p value:", p_value_result["p_value"])
    print("n valid permutations:", p_value_result["n_valid_permutations"])
    print("Saved:", permutation_path)
    print("Saved:", p_value_path)
    print("Saved:", null_csv_path)


[highIAT] permutation N=461, n_edges=22791, covariates=9
Running 5000 permutations for pearson_r with n_jobs=20
使用 n_jobs=20 并行计算 5000 次置换。


/Users/fmri24/fsl/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Observed score: 0.2651731290463208
Permutation p value: 0.003999200159968006
n valid permutations: 5000
Saved: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_Permutation/CPM_nestedCV_highIAT_permutation.pkl
Saved: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_Permutation/CPM_nestedCV_highIAT_permutation_p_value.pkl
Saved: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_Permutation/CPM_nestedCV_highIAT_permutation_null_scores.csv

[lowIAT] permutation N=653, n_edges=22791, covariates=10
Running 5000 permutations for pearson_r with n_jobs=20
使用 n_jobs=20 并行计算 5000 次置换。


/Users/fmri24/fsl/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Observed score: 0.28458225307848883
Permutation p value: 0.0007998400319936012
n valid permutations: 5000
Saved: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_Permutation/CPM_nestedCV_lowIAT_permutation.pkl
Saved: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_Permutation/CPM_nestedCV_lowIAT_permutation_p_value.pkl
Saved: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_Permutation/CPM_nestedCV_lowIAT_permutation_null_scores.csv


## 5. 输出文件

每个 subgroup 保存三个 permutation 结果文件：

- `CPM_nestedCV_<subgroup>_permutation.pkl`：`cpm_permutation_test_parallel()` 返回的原始 dict，包括 `null_scores`；
- `CPM_nestedCV_<subgroup>_permutation_p_value.pkl`：`calculate_permutation_p_value()` 返回的 p 值结果；
- `CPM_nestedCV_<subgroup>_permutation_null_scores.csv`：便于快速查看或画图的 permutation null scores。